# MiBici · exploración reproducible del equipo 27
Ejecutar después de `make data` y `make features`, con el entorno del proyecto.
Este notebook consume el manifiesto y el panel agregado; las transformaciones reutilizables están en `mibici`, no en el notebook.
Las cifras se obtienen al ejecutar; no hay resultados reales precargados.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'mibici' / 'config.py').exists())
sys.path.insert(0, str(root))
from mibici.config import DATA_MANIFEST_PATH, PROCESSED_DATA_DIR, FIN_ENTRENAMIENTO, FIN_VALIDACION
from mibici.preparar import verificar_datos
from mibici.features import filtrar_periodo, construir_perfil_estaciones
manifest = verificar_datos()
display(pd.Series(manifest['resumen'], name='valor'))
print('Versión de datos:', manifest['data_version'])

## Calidad y volumen
Revisar el porcentaje de descartes por mes, duplicados y estaciones ausentes. Un cero de demanda no prueba que una estación estuviera operativa. La nomenclatura de septiembre de 2026 no aporta fechas históricas de alta/baja.

In [ ]:
mensual = pd.DataFrame({k:v for k,v in manifest['archivos'].items() if k != 'nomenclatura'}).T
display(mensual[['originales', 'invalidos', 'fuera_periodo', 'duplicados', 'validos', 'codificacion']])
mensual['validos'].astype(int).plot.bar(figsize=(10,3), title='Viajes válidos por mes')
plt.ylabel('Viajes'); plt.tight_layout()

In [ ]:
panel = pd.read_parquet(PROCESSED_DATA_DIR / 'panel_estacion_hora.parquet')
print('Filas del panel:', len(panel), 'Estaciones:', panel.estacion_id.nunique())
print('Periodo:', panel.periodo.min(), 'a', panel.periodo.max())
print('Memoria del panel (MiB):', panel.memory_usage(deep=True).sum() / 2**20)
print('Clave estación/hora duplicada:', panel.duplicated(['estacion_id','periodo']).sum())
display(panel[['salidas','llegadas','capacidad','flujo_acumulado_dia_rel']].describe())
panel.groupby('hora_del_dia').salidas.mean().plot(figsize=(8,3), title='P1 · salidas medias por estación y hora')
plt.ylabel('Viajes'); plt.tight_layout()

## P2: prevalencia y particiones
La etiqueta mide magnitud del flujo neto total de tres horas respecto a la capacidad; no representa inventario observado. Los cortes excluyen ventanas que invaden la siguiente partición.

In [ ]:
val_inicio = str((pd.Timestamp(FIN_ENTRENAMIENTO) + pd.Timedelta(days=1)).date())
test_inicio = str((pd.Timestamp(FIN_VALIDACION) + pd.Timedelta(days=1)).date())
ultimo = str((pd.Timestamp(manifest['fin_exclusivo']) - pd.Timedelta(days=1)).date())
elegibles = panel.dropna(subset=['objetivo_en_riesgo','salidas_rezago_168'])
partes = {
 'train': filtrar_periodo(elegibles, fin=FIN_ENTRENAMIENTO, col_fin_objetivo='objetivo_fin_periodo'),
 'validation': filtrar_periodo(elegibles, inicio=val_inicio, fin=FIN_VALIDACION, col_fin_objetivo='objetivo_fin_periodo'),
 'test': filtrar_periodo(elegibles, inicio=test_inicio, fin=ultimo, col_fin_objetivo='objetivo_fin_periodo')
}
display(pd.DataFrame({k:{'filas':len(v), 'positivos':v.objetivo_en_riesgo.sum(), 'proporcion':v.objetivo_en_riesgo.mean(), 'inicio':v.periodo.min(), 'fin':v.periodo.max()} for k,v in partes.items()}).T)
print('Filas sin capacidad válida:', panel.capacidad.isna().sum())

## P3: perfiles construidos exclusivamente con entrenamiento
Las coordenadas se reservan para mapas. El escalado y K-Means se ejecutan desde `modeling/train.py`; las métricas de agrupamiento no se confunden con evaluación supervisada en test.

In [ ]:
nom = pd.read_parquet(PROCESSED_DATA_DIR / 'nomenclatura.parquet')
perfil = construir_perfil_estaciones(filtrar_periodo(panel, fin=FIN_ENTRENAMIENTO), nom)
display(perfil[['estacion_id','salidas_promedio_dia','balance_llegadas_salidas','latitud','longitud']].describe())
print('Estaciones sin salidas en train:', (perfil.salidas_promedio_dia == 0).sum())
print('Estaciones sin coordenadas:', perfil[['latitud','longitud']].isna().any(axis=1).sum())
perfil.filter(like='perfil_lab_h').mean().plot(figsize=(9,3), title='P3 · perfil horario medio laborable')
plt.ylabel('Proporción de salidas'); plt.tight_layout()

## Interpretación a completar después de ejecutar
Registrar los meses de mayor volumen, las tasas reales de descarte y de la etiqueta de P2, y los límites observados de cobertura. No modificar hiperparámetros a partir de resultados de prueba. Las conclusiones operativas son hipótesis sin inventario ni costos de rebalanceo.